In [ ]:
# SUIT NB03 + HMI SHARP exact-AR reprojection time series
# Date: 2026-02-04
# Targets: AR13842, AR13848, AR13841
#
# What this does:
# 1) Resolves NOAA AR -> HARPNUM for the requested day
# 2) Downloads definitive HMI SHARP magnetogram+bitmap for each AR/time
# 3) Downloads full-disk HMI magnetogram for the same times
# 4) Reprojects SHARP bitmap onto each SUIT frame for exact AR masking
# 5) Computes raw SUIT DN time series for each AR
# 6) Computes quiet-Sun DN from a fixed box around (0,0) using |B| < 20 G

In [ ]:
#Product: 0204SRS.txt
#Issued: 2026 Feb 04 0030 UTC
# Prepared jointly by the U.S. Dept. of Commerce, NOAA,
# Space Weather Prediction Center and the U.S. Air Force.
#
#Joint USAF/NOAA Solar Region Summary
#SRS Number 35 Issued at 0030Z on 04 Feb 2026
#Report compiled from data received at SWO on 03 Feb
#I.  Regions with Sunspots.  Locations Valid at 03/2400Z
#Nmbr Location  Lo  Area  Z   LL   NN Mag Type
#4358 N16W30   241  0030 Cro  04   05 Beta
#4362 S16E06   206  0040 Cao  05   07 Beta-Gamma
#4366 N14E07   204  1100 Fkc  16   42 Beta-Gamma-Delta
#4367 N09E38   174  0050 Cao  08   10 Beta
#4368 S10E38   174  0040 Hsx  02   01 Alpha
#4369 S03E43   168  0040 Hsx  02   01 Alpha
#4370 S18E50   161  0030 Cro  01   01 Beta
#4371 S22E60   151  0040 Dao  09   08 Beta
#4372 S24W39   250  0015 Bxo  04   06 Beta
#IA. H-alpha Plages without Spots.  Locations Valid at 03/2400Z Feb
#Nmbr  Location  Lo
#4359  N22W51   262
#4360  S16W29   240
#4363  S26E03   208
#4364  S17W89   300
#4365  N26W67   278
#II. Regions Due to Return 04 Feb to 06 Feb
#Nmbr Lat    Lo
#4343 S09    113
#4348 S17    083

In [ ]:
import os
import re
import glob
import urllib.request
import warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.time import Time
import sunpy.map
import drms

warnings.filterwarnings("ignore")

# ---------------------------
# USER SETTINGS
# ---------------------------
DATE_STR = "2026-02-04"
TARGET_NOAA = [14366, 14362, 14367, 14369]
JSOC_EMAIL = "captrucker86@gmail.com"
SUIT_GLOB = "/content/*NB03.fits"
OUTDIR = Path("./sharp_suit_2026_02_04")
OUTDIR.mkdir(parents=True, exist_ok=True)

QS_B_THRESHOLD = 20.0       # Gauss
QS_HALF_BOX = 50            # 100x100 pixel quiet-Sun box around (0,0)
MAX_TIME_DELTA_MIN = 20     # nearest SHARP/HMI time allowed for each SUIT frame

# ---------------------------
# HELPERS
# ---------------------------
def parse_time_str(t):
    if t is None:
        return None
    t = str(t).strip()
    t = t.replace("Z", "")
    t = t.split(".")[0]
    return pd.to_datetime(t, utc=False)

def suit_obs_time(fits_path):
    with fits.open(fits_path) as hdul:
        hdr = hdul[0].header
        t = hdr.get("T_OBS", hdr.get("DATE-OBS", None))
    if t is None:
        raise ValueError(f"No T_OBS/DATE-OBS in {fits_path}")
    t = str(t).replace("Z", "")
    t = t.split(".")[0]
    return pd.to_datetime(t)

def drms_to_datetime(t_rec):
    s = str(t_rec).replace("_TAI", "")
    return pd.to_datetime(s, format="%Y.%m.%d_%H:%M:%S")

def contains_noaa(noaa_ars, target):
    if pd.isna(noaa_ars):
        return False
    toks = re.split(r"[,\s]+", str(noaa_ars).strip())
    return str(target) in toks

def safe_basename_from_url(url):
    return url.split("/")[-1].split("?")[0]

def export_record(client, record, segments, outdir):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    segs = ",".join(segments)
    q = f"{record}{{{segs}}}"
    exp = client.export(q, method="url", protocol="fits")
    if hasattr(exp, "wait"):
        exp.wait()

    table = None
    if hasattr(exp, "urls") and exp.urls is not None:
        table = exp.urls
    elif hasattr(exp, "data") and exp.data is not None:
        table = exp.data
    else:
        raise RuntimeError(f"Could not get URL table for export: {q}")

    paths = {}
    for _, row in table.iterrows():
        url = row["url"] if "url" in row else None
        fname = row["filename"] if "filename" in row else safe_basename_from_url(url)
        local = outdir / fname
        if (url is not None) and (not local.exists()):
            urllib.request.urlretrieve(url, local)

        low = fname.lower()
        seg_name = None
        for seg in segments:
            if seg.lower() in low:
                seg_name = seg.lower()
                break
        if seg_name is None:
            seg_name = Path(fname).stem.split(".")[-1].lower()

        paths[seg_name] = str(local)

    return paths

def world_center_pixel(smap):
    center = SkyCoord(0*u.arcsec, 0*u.arcsec, frame=smap.coordinate_frame)
    xpix, ypix = smap.world_to_pixel(center)
    return int(np.round(xpix.value)), int(np.round(ypix.value))

def box_mask(shape, xc, yc, half):
    ny, nx = shape
    x0 = max(0, xc - half)
    x1 = min(nx, xc + half)
    y0 = max(0, yc - half)
    y1 = min(ny, yc + half)
    m = np.zeros(shape, dtype=bool)
    m[y0:y1, x0:x1] = True
    return m

def nearest_row(df, t):
    dt = np.abs((df["T_REC_DT"] - t).dt.total_seconds()) / 60.0
    i = dt.idxmin()
    if dt.loc[i] > MAX_TIME_DELTA_MIN:
        return None
    return df.loc[i]

def raw_stats(arr, mask):
    vals = arr[mask]
    vals = vals[np.isfinite(vals)]
    vals = vals[vals > 0]
    if vals.size == 0:
        return np.nan, np.nan, 0
    return float(np.mean(vals)), float(np.median(vals)), int(vals.size)

# ---------------------------
# LOAD SUIT FILES
# ---------------------------
suit_files = sorted(glob.glob(SUIT_GLOB))
if len(suit_files) == 0:
    raise FileNotFoundError(f"No SUIT files matched: {SUIT_GLOB}")

suit_rows = []
for f in suit_files:
    t = suit_obs_time(f)
    if str(t.date()) == DATE_STR:
        suit_rows.append({"file": f, "time": t})

suit_df = pd.DataFrame(suit_rows).sort_values("time").reset_index(drop=True)
if suit_df.empty:
    raise RuntimeError(f"No SUIT files found on {DATE_STR}")

print(f"SUIT frames on {DATE_STR}: {len(suit_df)}")

# ---------------------------
# QUERY SHARP DAY TABLE
# ---------------------------
c = drms.Client(email=JSOC_EMAIL)

day_query = f"hmi.sharp_720s[][2026.02.04_00:00:00_TAI/1d@720s]"
keys = ["HARPNUM", "T_REC", "NOAA_ARS"]
day_tbl = c.query(day_query, key=keys, rec_index=True)

if day_tbl is None or len(day_tbl) == 0:
    raise RuntimeError("No SHARP records returned for the requested day.")

day_tbl = day_tbl.copy()
day_tbl["T_REC_DT"] = day_tbl["T_REC"].apply(drms_to_datetime)

# ---------------------------
# RESOLVE NOAA -> HARPNUM
# ---------------------------
harp_info = {}

for noaa in TARGET_NOAA:
    sub = day_tbl[day_tbl["NOAA_ARS"].apply(lambda s: contains_noaa(s, noaa))].copy()
    if sub.empty:
        raise RuntimeError(f"NOAA AR {noaa} not found in hmi.sharp_720s on {DATE_STR}")
    harpnum = int(sub["HARPNUM"].mode().iloc[0])
    sub = sub[sub["HARPNUM"] == harpnum].sort_values("T_REC_DT").reset_index(drop=True)

    harp_info[noaa] = {
        "harpnum": harpnum,
        "table": sub
    }
    print(f"NOAA AR {noaa} -> HARPNUM {harpnum} ({len(sub)} records)")

# ---------------------------
# DOWNLOAD ON DEMAND CACHE
# ---------------------------
sharp_cache = {}
fd_cache = {}

def get_sharp_paths(noaa, t_rec):
    key = (noaa, str(t_rec))
    if key in sharp_cache:
        return sharp_cache[key]

    harpnum = harp_info[noaa]["harpnum"]
    rec = f"hmi.sharp_720s[{harpnum}][{t_rec}]"
    out = OUTDIR / f"AR{noaa}"
    paths = export_record(c, rec, ["magnetogram", "bitmap"], out)
    sharp_cache[key] = paths
    return paths

def get_full_disk_hmi_paths(t_rec):
    key = str(t_rec)
    if key in fd_cache:
        return fd_cache[key]

    rec = f"hmi.M_720s[{t_rec}]"
    out = OUTDIR / "HMI_full_disk"
    paths = export_record(c, rec, ["magnetogram"], out)
    fd_cache[key] = paths
    return paths

# ---------------------------
# MAIN LOOP
# ---------------------------
records = []
preview_frames = []

for i, row in suit_df.iterrows():
    suit_file = row["file"]
    t_suit = row["time"]
    suit_map = sunpy.map.Map(suit_file)
    suit_data = np.asarray(suit_map.data, dtype=np.float32)

    rec = {
        "time": t_suit,
        "suit_file": os.path.basename(suit_file)
    }

    # Use the first AR's nearest SHARP time to fetch the full-disk HMI
    # (12-min cadence, so all target ARs should align very closely)
    first_noaa = TARGET_NOAA[0]
    r0 = nearest_row(harp_info[first_noaa]["table"], t_suit)
    if r0 is None:
        print(f"Skipping {t_suit} : no SHARP record within {MAX_TIME_DELTA_MIN} min")
        continue

    t_rec0 = r0["T_REC"]
    fd_paths = get_full_disk_hmi_paths(t_rec0)
    hmi_fd_map = sunpy.map.Map(fd_paths["magnetogram"])
    hmi_fd_on_suit = hmi_fd_map.reproject_to(suit_map.wcs)

    # Quiet Sun at disk center from full-disk HMI
    xc_qs, yc_qs = world_center_pixel(suit_map)
    qs_box = box_mask(suit_data.shape, xc_qs, yc_qs, QS_HALF_BOX)
    fdB = np.asarray(hmi_fd_on_suit.data, dtype=np.float32)

    qs_mask = (
        qs_box &
        np.isfinite(fdB) &
        (np.abs(fdB) < QS_B_THRESHOLD) &
        np.isfinite(suit_data) &
        (suit_data > 0)
    )

    qs_mean, qs_median, qs_np = raw_stats(suit_data, qs_mask)
    rec["QS_mean_DN"] = qs_mean
    rec["QS_median_DN"] = qs_median
    rec["QS_npix"] = qs_np
    rec["HMI_ref_trec"] = str(t_rec0)

    frame_info = {
        "time": t_suit,
        "suit_file": suit_file,
        "boxes": {},
        "qs_center_pix": (xc_qs, yc_qs)
    }

    for noaa in TARGET_NOAA:
        row_near = nearest_row(harp_info[noaa]["table"], t_suit)
        if row_near is None:
            rec[f"AR{noaa}_mean_DN"] = np.nan
            rec[f"AR{noaa}_median_DN"] = np.nan
            rec[f"AR{noaa}_npix"] = 0
            rec[f"AR{noaa}_trec"] = None
            continue

        t_rec = row_near["T_REC"]
        rec[f"AR{noaa}_trec"] = str(t_rec)

        sharp_paths = get_sharp_paths(noaa, t_rec)
        sharp_mag_map = sunpy.map.Map(sharp_paths["magnetogram"])
        sharp_bmp_map = sunpy.map.Map(sharp_paths["bitmap"])

        sharp_mag_on_suit = sharp_mag_map.reproject_to(suit_map.wcs)
        sharp_bmp_on_suit = sharp_bmp_map.reproject_to(suit_map.wcs)

        bmp = np.asarray(sharp_bmp_on_suit.data, dtype=np.float32)
        shpB = np.asarray(sharp_mag_on_suit.data, dtype=np.float32)

        # BITMAP values inside the SHARP smooth bounding curve are 33/34
        ar_mask = (
            np.isfinite(bmp) &
            (bmp >= 33) &
            np.isfinite(shpB) &
            np.isfinite(suit_data) &
            (suit_data > 0)
        )

        ar_mean, ar_median, ar_np = raw_stats(suit_data, ar_mask)
        rec[f"AR{noaa}_mean_DN"] = ar_mean
        rec[f"AR{noaa}_median_DN"] = ar_median
        rec[f"AR{noaa}_npix"] = ar_np

        if np.isfinite(ar_mean) and np.isfinite(qs_mean) and qs_mean > 0:
            rec[f"AR{noaa}_over_QS"] = ar_mean / qs_mean
        else:
            rec[f"AR{noaa}_over_QS"] = np.nan

        ys, xs = np.where(ar_mask)
        if len(xs) > 0:
            x0, x1 = int(xs.min()), int(xs.max())
            y0, y1 = int(ys.min()), int(ys.max())
        else:
            x0 = x1 = y0 = y1 = None

        frame_info["boxes"][f"AR{noaa}"] = {
            "bbox_pix": (x0, x1, y0, y1),
            "npix": ar_np
        }

    records.append(rec)
    preview_frames.append(frame_info)

    print(
        f"{t_suit.strftime('%H:%M:%S')}  "
        f"QS={qs_mean:.1f} DN  "
        + "  ".join(
            [
                f"AR{n}={rec.get(f'AR{n}_mean_DN', np.nan):.1f}"
                for n in TARGET_NOAA
            ]
        )
    )

# ---------------------------
# SAVE TABLE
# ---------------------------
df = pd.DataFrame(records).sort_values("time").reset_index(drop=True)
csv_path = OUTDIR / f"suit_nb03_sharp_timeseries_{DATE_STR}.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved CSV -> {csv_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

# -------------------------
# Read CSV directly
# -------------------------
csv_path = "/content/sharp_suit_2026_02_04/suit_nb03_sharp_timeseries_2026-02-04.csv"
df = pd.read_csv(csv_path)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

# Reference time near 07:00:23 UTC for title
target_ref = pd.Timestamp("2026-02-04 07:00:23")
ref_time = df.loc[(df["time"] - target_ref).abs().idxmin(), "time"]

# -------------------------
# Plot styling
# -------------------------
mean_series = [
    ("AR14366_mean_DN", "AR14366 (βγδ)", "#f06449", "o"),
    ("AR14362_mean_DN", "AR14362 (βγ)",  "#e5c100", "s"),
    ("AR14367_mean_DN", "AR14367 (β)",   "#39b54a", "^"),
    ("AR14369_mean_DN", "AR14369 (α)",   "#20b7ea", "D"),
]

median_series = [
    ("AR14366_median_DN", "AR14366 (βγδ)", "#f06449", "o"),
    ("AR14362_median_DN", "AR14362 (βγ)",  "#e5c100", "s"),
    ("AR14367_median_DN", "AR14367 (β)",   "#39b54a", "^"),
    ("AR14369_median_DN", "AR14369 (α)",   "#20b7ea", "D"),
]

qs_mean = ("QS_mean_DN", "QS centre", "#39e7ef", "*", "--")
qs_median = ("QS_median_DN", "QS centre", "#39e7ef", "*", "--")

def kfmt(x, pos):
    return f"{x/1000:.2f}k"

fig, axes = plt.subplots(
    2, 1, figsize=(13.5, 8.6), sharex=True,
    gridspec_kw={"hspace": 0.06}
)

fig.patch.set_facecolor("#e9e9e9")

for ax in axes:
    ax.set_facecolor("#f1f1f1")
    ax.grid(True, which="major", color="#bdbdbd", alpha=0.35, linewidth=0.8)
    ax.yaxis.set_major_formatter(FuncFormatter(kfmt))
    for spine in ax.spines.values():
        spine.set_color("#444444")
    ax.tick_params(axis="both", labelsize=10, colors="#222222")

# -------------------------
# Top panel: Mean
# -------------------------
for col, label, color, marker in mean_series:
    ax = axes[0]
    ax.plot(
        df["time"], df[col],
        color=color, marker=marker, markersize=5.5,
        linewidth=1.8, label=label
    )

axes[0].plot(
    df["time"], df[qs_mean[0]],
    color=qs_mean[2], marker=qs_mean[3], linestyle=qs_mean[4],
    markersize=7, linewidth=1.5, label=qs_mean[1]
)

axes[0].set_ylabel("Mean Intensity (DN)", fontsize=11)
axes[0].legend(
    loc="upper left", fontsize=9,
    frameon=True, facecolor="#f0f0f0", edgecolor="#bbbbbb"
)

# -------------------------
# Bottom panel: Median
# -------------------------
for col, label, color, marker in median_series:
    ax = axes[1]
    ax.plot(
        df["time"], df[col],
        color=color, marker=marker, markersize=5.5,
        linewidth=1.8, label=label
    )

axes[1].plot(
    df["time"], df[qs_median[0]],
    color=qs_median[2], marker=qs_median[3], linestyle=qs_median[4],
    markersize=7, linewidth=1.5, label=qs_median[1]
)

axes[1].set_ylabel("Median Intensity (DN)", fontsize=11)
axes[1].set_xlabel("Time UTC — 4 Feb 2026", fontsize=11)
axes[1].legend(
    loc="upper left", fontsize=9,
    frameon=True, facecolor="#f0f0f0", edgecolor="#bbbbbb"
)

# X-axis formatting
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.setp(axes[1].get_xticklabels(), rotation=35, ha="right")

# Figure title
fig.suptitle(
    "SUIT NB03 (Mg II k, 2796 Å) — 4 Feb 2026 ARs [raw DN]\n"
    f"Exact SHARP masks | QS centre at (0,0), |B|<20 G | Ref: {ref_time:%Y-%m-%dT%H:%M:%S} UTC",
    fontsize=13, fontweight="bold", y=0.985
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

out_png = "suit_nb03_sharp_timeseries_image3_style.png"
plt.savefig(out_png, dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "/content/suit_nb03_sharp_timeseries_totalfeatures_2026-02-04.csv"
df = pd.read_csv(csv_path)

series = {
    14366: {"label": "AR14366 (βγδ)", "color": "#f06449", "marker": "o"},
    14362: {"label": "AR14362 (βγ)",  "color": "#e5c100", "marker": "s"},
    14367: {"label": "AR14367 (β)",   "color": "#39b54a", "marker": "^"},
    14369: {"label": "AR14369 (α)",   "color": "#20b7ea", "marker": "D"},
}

fig, axes = plt.subplots(4, 1, figsize=(13, 19))
fig.patch.set_facecolor("#e9e9e9")

for ax in axes:
    ax.set_facecolor("#f2f2f2")
    ax.grid(True, alpha=0.28)
    for spine in ax.spines.values():
        spine.set_color("#444444")
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))

# 1) Intensity(DN) vs Total Magnetic Flux
for ar, st in series.items():
    axes[0].scatter(
        df[f"AR{ar}_TotalMagneticFlux"],
        df[f"AR{ar}_median_DN"],
        color=st["color"],
        marker=st["marker"],
        s=55,
        edgecolors="black",
        linewidths=0.4,
        alpha=0.9,
        label=st["label"]
    )
axes[0].set_xlabel("Total Magnetic Flux")
axes[0].set_ylabel("Intensity (DN)")
axes[0].set_title("Intensity (DN) vs Total Magnetic Flux", fontsize=12, fontweight="bold")
axes[0].legend(loc="best", fontsize=9, frameon=True)

# 2) Intensity(DN) vs Total Magnetic Energy
for ar, st in series.items():
    axes[1].scatter(
        df[f"AR{ar}_TotalMagneticEnergy"],
        df[f"AR{ar}_median_DN"],
        color=st["color"],
        marker=st["marker"],
        s=55,
        edgecolors="black",
        linewidths=0.4,
        alpha=0.9,
        label=st["label"]
    )
axes[1].set_xlabel("Total Magnetic Energy")
axes[1].set_ylabel("Intensity (DN)")
axes[1].set_title("Intensity (DN) vs Total Magnetic Energy", fontsize=12, fontweight="bold")
axes[1].legend(loc="best", fontsize=9, frameon=True)

# 3) Intensity(DN) vs Total Helicity
for ar, st in series.items():
    axes[2].scatter(
        df[f"AR{ar}_TotalHelicity"],
        df[f"AR{ar}_median_DN"],
        color=st["color"],
        marker=st["marker"],
        s=55,
        edgecolors="black",
        linewidths=0.4,
        alpha=0.9,
        label=st["label"]
    )
axes[2].set_xlabel("Total Helicity")
axes[2].set_ylabel("Intensity (DN)")
axes[2].set_title("Intensity (DN) vs Total Helicity", fontsize=12, fontweight="bold")
axes[2].legend(loc="best", fontsize=9, frameon=True)

# 4) Intensity(DN) vs Total Free Energy
for ar, st in series.items():
    axes[3].scatter(
        df[f"AR{ar}_TotalFreeEnergy"],
        df[f"AR{ar}_median_DN"],
        color=st["color"],
        marker=st["marker"],
        s=55,
        edgecolors="black",
        linewidths=0.4,
        alpha=0.9,
        label=st["label"]
    )
axes[3].set_xlabel("Total Free Energy")
axes[3].set_ylabel("Intensity (DN)")
axes[3].set_title("Intensity (DN) vs Total Free Energy", fontsize=12, fontweight="bold")
axes[3].legend(loc="best", fontsize=9, frameon=True)

plt.suptitle(
    "Intensity (DN) on Y-axis and magnetic parameters on X-axis",
    fontsize=14, fontweight="bold", y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])

out_png = "/content/intensity_vs_total_magnetic_parameters_with_free_energy_scatter_2026-02-04.png"
plt.savefig(out_png, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved plot: {out_png}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "/content/suit_nb03_sharp_timeseries_totalfeatures_2026-02-04.csv"
df = pd.read_csv(csv_path)

sunspots = {
    14366: {"label": "AR14366 (βγδ)", "color": "#f06449", "marker": "o"},
    14362: {"label": "AR14362 (βγ)",  "color": "#e5c100", "marker": "s"},
    14367: {"label": "AR14367 (β)",   "color": "#39b54a", "marker": "^"},
    14369: {"label": "AR14369 (α)",   "color": "#20b7ea", "marker": "D"},
}

params = [
    ("TotalMagneticFlux", "Total Magnetic Flux"),
    ("TotalMagneticEnergy", "Total Magnetic Energy"),
    ("TotalHelicity", "Total Helicity"),
    ("TotalFreeEnergy", "Total Free Energy"),
]

for ar, st in sunspots.items():
    for suffix, title in params:
        plt.figure(figsize=(4, 4), facecolor="#e9e9e9")
        ax = plt.gca()
        ax.set_facecolor("#f2f2f2")
        ax.grid(True, alpha=0.28)
        for spine in ax.spines.values():
            spine.set_color("#444444")

        plt.scatter(
            df[f"AR{ar}_{suffix}"],
            df[f"AR{ar}_median_DN"],
            color=st["color"],
            marker=st["marker"],
            s=45,
            edgecolors="black",
            linewidths=0.4,
            alpha=0.9
        )

        plt.xlabel(title)
        plt.ylabel("Intensity (DN)")
        plt.title(f"{st['label']}: Intensity vs {title}", fontweight="bold")
        plt.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))

        out_png = f"/content/{ar}_{suffix}_scatter.png"
        plt.tight_layout()
        plt.savefig(out_png, dpi=220, bbox_inches="tight", facecolor="#e9e9e9")
        plt.show()

        print(f"Saved: {out_png}")